# Clonamos el repositorio para obtener el dataset

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/joanby/deeplearning-az.git
    %cd 'deeplearning-az/updated/Part 2 - Convolutional Neural Networks (CNN)'

# Parte 1 - Construir el modelo de CNN

# Importar las librerías y paquetes

In [5]:
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import Dense

# Inicializar la CNN

In [4]:
classifier = Sequential()


# Paso 1 - Convolución

In [ ]:
classifier.add(Conv2D(filters = 32,kernel_size = (3, 3), 
                      input_shape = (64, 64, 3), activation = "relu"))


# Paso 2 - Max Pooling

In [7]:
classifier.add(MaxPooling2D(pool_size = (2,2)))

# Una segunda capa de convolución y max pooling

In [8]:
classifier.add(Conv2D(filters = 32,kernel_size = (3, 3), activation = "relu"))

In [9]:
classifier.add(MaxPooling2D(pool_size = (2,2)))


# Paso 3 - Flattening

In [10]:
classifier.add(Flatten())

# Paso 4 - Full Connection

In [11]:
classifier.add(Dense(units = 128, activation = "relu"))
classifier.add(Dense(units = 1, activation = "sigmoid"))

# Compilar la CNN
# Como va a ser entrenada?

In [12]:
classifier.compile(optimizer = "adam", loss = "binary_crossentropy", metrics = ["accuracy"])


# Parte 2 - Ajustar la CNN a las imágenes para entrenar 

In [ ]:
import tensorflow as tf

rescale = tf.keras.layers.Rescaling(1./255)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomShear(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomFlip('horizontal'),
])

training_dataset = tf.keras.utils.image_dataset_from_directory('dataset/training_set',
                                                    image_size=(64, 64),
                                                    batch_size=32,
                                                    label_mode='binary')
class_names = training_dataset.class_names
training_dataset = training_dataset.map(lambda x, y: (data_augmentation(rescale(x), training=True), y))

testing_dataset = tf.keras.utils.image_dataset_from_directory('dataset/test_set',
                                                image_size=(64, 64),
                                                batch_size=32,
                                                label_mode='binary')
testing_dataset = testing_dataset.map(lambda x, y: (rescale(x), y))

classifier.fit(x=training_dataset,
              validation_data=testing_dataset,
              epochs=25)



# Parte 3 - Cómo hacer nuevas predicciones

In [ ]:
import numpy as np
import keras.utils as image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = classifier.predict(test_image)
class_names  # equivalente al antiguo training_dataset.class_indices
if result[0][0] == 1:
    prediction = 'dog'
else:
    prediction = 'cat'

